# AI Functions: Customer Experience Telemetry

Turn raw chat threads, call transcripts, and support tickets into structured sentiment and topic telemetry — entirely in SQL with Snowflake AI Functions.

### What You'll Learn

- Score sentiment with `AI_SENTIMENT` (overall + per-aspect)
- Model topics with `AI_CLASSIFY`
- Extract structured fields with `AI_EXTRACT`
- Discover themes across thousands of rows with `AI_AGG` / `AI_SUMMARIZE_AGG`
- Surface churn-risk conversations with `AI_FILTER`
- Assemble a governed CX telemetry table and point AI Function Studio at a custom function

**Prerequisites:** Run `lab/setup.sql`, then `python lab/data_gen.py`, before starting this notebook.

---
## Section 1: Connect & Explore

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE FIELD_CX_DEMO").collect()
session.sql("USE SCHEMA AI_FUNCTIONS").collect()
session.sql("USE WAREHOUSE CX_AI_FUNCTIONS_WH").collect()
print("Connected to FIELD_CX_DEMO.AI_FUNCTIONS")

In [ ]:
# Peek at the raw conversations (loaded by data_gen.py)
session.sql("""
    SELECT thread_id, customer_id, channel, transcript
    FROM chat_threads
    LIMIT 5
""").show(max_width=120)

---
## Section 2: Sentiment with AI_SENTIMENT

`AI_SENTIMENT(text, categories)` returns overall sentiment plus per-aspect sentiment in one call.

In [ ]:
# Overall + aspect sentiment for a sample of threads
session.sql("""
    SELECT
        thread_id,
        AI_SENTIMENT(transcript,
            ['valuation_accuracy','pricing_billing','onboarding']) AS sentiment
    FROM chat_threads
    LIMIT 5
""").show(max_width=140)

In [ ]:
# Materialize overall sentiment as a column for trending
session.sql("""
    CREATE OR REPLACE TABLE thread_sentiment AS
    SELECT
        thread_id,
        customer_id,
        created_at,
        AI_SENTIMENT(transcript):categories[0]:sentiment::string AS overall_sentiment
    FROM chat_threads
""").collect()

session.sql("""
    SELECT overall_sentiment, COUNT(*) AS threads
    FROM thread_sentiment
    GROUP BY overall_sentiment
    ORDER BY threads DESC
""").show()

---
## Section 3: Topic Modeling with AI_CLASSIFY

Map each conversation to a support taxonomy. Use `{'output_mode': 'multi'}` for threads that span topics.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE classified_threads AS
    SELECT
        thread_id,
        customer_id,
        created_at,
        transcript,
        AI_CLASSIFY(transcript,
            ['valuation_accuracy','pricing_billing','onboarding',
             'bug_report','cancellation','feature_request'],
            {'task_description': 'Support topics for a home-valuation product'}
        ):labels[0]::string AS topic
    FROM chat_threads
""").collect()

session.sql("""
    SELECT topic, COUNT(*) AS threads
    FROM classified_threads
    GROUP BY topic
    ORDER BY threads DESC
""").show()

---
## Section 4: Structured Fields with AI_EXTRACT

Pull named fields out of free-text support tickets. Ask one clear question per field.

In [ ]:
session.sql("""
    SELECT
        ticket_id,
        AI_EXTRACT(
            text => body,
            responseFormat => {
                'intent': 'What does the customer want to do?',
                'product_area': 'Which product area is mentioned?',
                'urgency': 'Is this urgent? yes or no'
            }
        ):response AS fields
    FROM support_tickets
    LIMIT 5
""").show(max_width=140)

---
## Section 5: Theme Discovery with AI_AGG & AI_SUMMARIZE_AGG

The "unsupervised" view: surface dominant themes across thousands of rows. Both support `GROUP BY`.

In [ ]:
# Most common complaints per topic across the whole corpus
session.sql("""
    SELECT
        topic,
        AI_AGG(transcript,
            'Identify the most common customer complaints in these conversations') AS complaints
    FROM classified_threads
    GROUP BY topic
""").show(max_width=160)

In [ ]:
# A single executive summary of all support tickets
session.sql("""
    SELECT AI_SUMMARIZE_AGG(body) AS ticket_summary
    FROM support_tickets
""").show(max_width=200)

---
## Section 6: At-Risk Detection with AI_FILTER

Keep only churn-risk / escalation conversations, then join to structured data to prioritize by customer.

In [ ]:
session.sql("""
    SELECT
        t.thread_id,
        c.customer_id,
        c.customer_type,
        c.plan,
        c.region
    FROM chat_threads t
    JOIN customers c ON c.customer_id = t.customer_id
    WHERE AI_FILTER('This customer is frustrated and at risk of cancelling: ' || t.transcript)
    LIMIT 20
""").show()

---
## Section 7: Assemble the CX Telemetry Table

Combine sentiment + topic into one governed table you can trend in BI and feed to a churn model.

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE cx_telemetry AS
    SELECT
        s.thread_id,
        s.customer_id,
        s.created_at,
        s.overall_sentiment,
        c.topic
    FROM thread_sentiment s
    JOIN classified_threads c ON c.thread_id = s.thread_id
""").collect()

# Sentiment mix by topic — the telemetry Product/CX teams have been missing
session.sql("""
    SELECT topic, overall_sentiment, COUNT(*) AS threads
    FROM cx_telemetry
    GROUP BY topic, overall_sentiment
    ORDER BY topic, threads DESC
""").show(30)

---
## Section 8: Optimize with AI Function Studio

When a built-in isn't enough, define a custom function on `AI_COMPLETE`, then use **AI Function Studio** (Snowsight → AI & ML → Cortex AI Function Studio) to compare prompts/models on a labeled sample and review accuracy vs cost.

The cell below shows a custom escalation score you might build, evaluate, and tune in Studio.

---
## Summary

You built a customer-experience telemetry pipeline entirely in SQL:

- **AI_SENTIMENT** → overall and per-aspect sentiment
- **AI_CLASSIFY** → topic taxonomy
- **AI_EXTRACT** → structured fields from free text
- **AI_AGG / AI_SUMMARIZE_AGG** → theme discovery across the corpus
- **AI_FILTER** → churn-risk conversations, joined to customers
- **AI Function Studio** → optimize a custom escalation score

The resulting `CX_TELEMETRY` table is consumed by the **Conversational BI** module, where a semantic view + Cortex Agent analyze it alongside churn and revenue. Swap the synthetic tables for your real chat logs and call transcripts — the SQL is unchanged.